# 第 07 天：价值因子 2

> 来自《30 天因子研究计划》第 7 天  
> 主题：价值因子 2  
> 必做：EP / BP  
> 选做：股息率  
> 目标产出：价值因子报告

---

## 0. 今天你要真正学会什么？

第 6 天我们从 PE、PB、EV/EBITDA 入手，学习了“估值倍数越低越便宜”。  
今天换一个更适合因子排序的视角：


EP = 盈利 / 价格
BP = 净资产 / 价格
股息率 = 分红 / 价格


这些指标天然更像因子：


越大，通常越便宜或现金回报越高


学完以后，你应该能回答：

1. EP 和 PE 是什么关系？
2. BP 和 PB 是什么关系？
3. 为什么股息率也是价值因子的一部分？
4. 如何构建价值综合分数？
5. 如何写一个简化价值因子报告？

一句话版：

> PE、PB 是“价格是基本面的多少倍”，EP、BP 是“每一元价格买到多少基本面”。

---

## 1. 从“倍数”到“收益率”

PE 的公式是：


PE = 市值 / 净利润


EP 的公式是：


EP = 净利润 / 市值 = 1 / PE


这很像一个收益率概念：

> 你花 1 元市值，背后对应多少净利润。

同理：


PB = 市值 / 净资产
BP = 净资产 / 市值 = 1 / PB


BP 越高，代表每 1 元市值背后对应的账面净资产越多。

---

## 2. 三个核心指标

### 2.1 EP：盈利收益率


EP = 净利润 / 总市值


直觉：

- EP 越高，盈利相对市值越高。
- 如果 PE = 10，那么 EP = 10%。
- 如果 PE = 20，那么 EP = 5%。

EP 的优点是方向天然正向：越大越便宜。  
风险是净利润为负时，EP 也会为负，需要谨慎处理。

### 2.2 BP：账面市值比


BP = 净资产 / 总市值


BP 是经典价值因子。Fama-French 的 HML 因子就和账面市值比有关。

直觉：

- BP 高：账面资产相对市值多，偏价值。
- BP 低：市场给了较高估值，可能偏成长。

风险是账面净资产的质量不一定可靠。

### 2.3 股息率


股息率 = 现金分红 / 总市值


直觉：

> 你用当前市值买入公司，一年能拿到多少现金分红回报。

股息率高的公司通常更成熟、现金流更稳定，但也可能代表增长不足。

---

## 3. 准备 Python 环境


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

rng = np.random.default_rng(20260705)


如果缺包，可以先安装：


In [ ]:
pip install numpy pandas matplotlib


---

## 4. 构造模拟基本面数据


In [ ]:
n = 600

df = pd.DataFrame({
    "ticker": [f"Stock_{i:03d}" for i in range(n)],
    "market_cap": rng.lognormal(mean=10.6, sigma=0.9, size=n),
    "net_income": rng.normal(loc=1000, scale=650, size=n),
    "book_equity": rng.normal(loc=6500, scale=2600, size=n),
})

# 分红通常和盈利、成熟度有关，但并非所有公司都分红
payout_ratio = np.clip(rng.normal(loc=0.25, scale=0.18, size=n), 0, 0.9)
df["cash_dividend"] = np.maximum(df["net_income"], 0) * payout_ratio

# 制造亏损和负净资产
df.loc[rng.choice(n, size=45, replace=False), "net_income"] *= -1
df.loc[rng.choice(n, size=18, replace=False), "book_equity"] *= -0.2

df.head()


字段：

- `market_cap`：总市值
- `net_income`：净利润
- `book_equity`：净资产
- `cash_dividend`：现金分红

---

## 5. 计算 EP、BP、股息率


In [ ]:
df["ep"] = df["net_income"] / df["market_cap"]
df["bp"] = df["book_equity"] / df["market_cap"]
df["dividend_yield"] = df["cash_dividend"] / df["market_cap"]

df[["ticker", "ep", "bp", "dividend_yield"]].head()


初步描述：


In [ ]:
df[["ep", "bp", "dividend_yield"]].describe()


你会发现 EP、BP 可能有负值。  
负 EP 代表亏损，负 BP 代表负净资产，这些样本要单独小心。

---

## 6. 指标清洗和标准化

### 6.1 工具函数


In [ ]:
def winsorize_series(s: pd.Series, lower: float = 0.01, upper: float = 0.99) -> pd.Series:
    lo = s.quantile(lower)
    hi = s.quantile(upper)
    return s.clip(lo, hi)


def zscore(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std()


def clean_positive_or_nan(s: pd.Series) -> pd.Series:
    return s.where(s > 0, np.nan)


### 6.2 构造价值因子

EP、BP、股息率天然是越大越偏价值，所以不需要取负号。


In [ ]:
df["ep_clean"] = clean_positive_or_nan(df["ep"])
df["bp_clean"] = clean_positive_or_nan(df["bp"])
df["dividend_yield_clean"] = df["dividend_yield"].where(df["dividend_yield"] >= 0, np.nan)

df["value_ep"] = zscore(winsorize_series(df["ep_clean"]))
df["value_bp"] = zscore(winsorize_series(df["bp_clean"]))
df["value_dividend"] = zscore(winsorize_series(df["dividend_yield_clean"]))

df["value_score"] = df[["value_ep", "value_bp", "value_dividend"]].mean(axis=1, skipna=True)

df[["ticker", "ep", "bp", "dividend_yield", "value_score"]].head()


解释：

- `value_ep` 高：盈利收益率高。
- `value_bp` 高：账面市值比高。
- `value_dividend` 高：股息率高。
- `value_score`：三者综合。

---

## 7. EP/BP 和 PE/PB 的关系

我们验证一下：


In [ ]:
df["pe"] = df["market_cap"] / df["net_income"]
df["pb"] = df["market_cap"] / df["book_equity"]

df.loc[df["pe"] <= 0, "pe"] = np.nan
df.loc[df["pb"] <= 0, "pb"] = np.nan

relation = df[["pe", "pb", "ep", "bp"]].corr()
relation


正常情况下：

- PE 和 EP 应该负相关。
- PB 和 BP 应该负相关。

也就是说，低 PE 对应高 EP，低 PB 对应高 BP。

---

## 8. 价值因子报告：先看分布


In [ ]:
factor_cols = ["value_ep", "value_bp", "value_dividend", "value_score"]
df[factor_cols].describe()


画分布：


In [ ]:
df[factor_cols].hist(bins=30, figsize=(10, 7))
plt.tight_layout()
plt.show()


看分布的目的：

- 是否有极端值。
- 是否大量缺失。
- 是否某个指标几乎没有区分度。

---

## 9. 价值因子报告：看相关性


In [ ]:
factor_corr = df[factor_cols].corr()
factor_corr


画热力图：


In [ ]:
plt.imshow(factor_corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(factor_cols)), factor_cols, rotation=45)
plt.yticks(range(len(factor_cols)), factor_cols)
plt.title("Value Factor Correlation")
plt.tight_layout()
plt.show()


相关性解释：

- EP 和 BP 相关性高：可能都在捕捉便宜。
- 股息率和 EP 相关：盈利好的成熟公司更可能分红。
- 相关性太高：合成时信息可能重复。
- 相关性较低：可能提供互补信息。

---

## 10. 模拟未来收益并做报告

为了演示报告流程，我们让综合价值分数对未来收益有一点正向影响。


In [ ]:
noise = rng.normal(0, 0.055, size=n)
df["future_20d_ret"] = 0.010 * df["value_score"].fillna(0) + noise

rank_ic = df["value_score"].corr(df["future_20d_ret"], method="spearman")
print("价值综合分数 Rank IC:", round(rank_ic, 4))


分组收益：


In [ ]:
valid = df.dropna(subset=["value_score", "future_20d_ret"]).copy()
valid["group"] = pd.qcut(
    valid["value_score"].rank(method="first"),
    q=5,
    labels=["G1 低价值", "G2", "G3", "G4", "G5 高价值"]
)

group_report = valid.groupby("group", observed=True).agg(
    stock_count=("ticker", "count"),
    avg_value_score=("value_score", "mean"),
    avg_future_20d_ret=("future_20d_ret", "mean"),
    avg_ep=("ep", "mean"),
    avg_bp=("bp", "mean"),
    avg_dividend_yield=("dividend_yield", "mean"),
)

group_report


画图：


In [ ]:
group_report["avg_future_20d_ret"].plot(kind="bar", title="价值分组未来收益")
plt.ylabel("Average future 20D return")
plt.xticks(rotation=30)
plt.show()


---

## 11. 今日目标产出：价值因子报告函数


In [ ]:
def build_value_report(data: pd.DataFrame) -> dict:
    factor_cols = ["value_ep", "value_bp", "value_dividend", "value_score"]
    valid = data.dropna(subset=["value_score", "future_20d_ret"]).copy()
    valid["group"] = pd.qcut(
        valid["value_score"].rank(method="first"),
        q=5,
        labels=["G1 低价值", "G2", "G3", "G4", "G5 高价值"]
    )

    summary = data[factor_cols].describe()
    corr = data[factor_cols].corr()
    ic = data["value_score"].corr(data["future_20d_ret"], method="spearman")
    group = valid.groupby("group", observed=True).agg(
        stock_count=("ticker", "count"),
        avg_value_score=("value_score", "mean"),
        avg_future_20d_ret=("future_20d_ret", "mean"),
        avg_ep=("ep", "mean"),
        avg_bp=("bp", "mean"),
        avg_dividend_yield=("dividend_yield", "mean"),
    )

    return {
        "factor_summary": summary,
        "factor_corr": corr,
        "rank_ic": ic,
        "group_report": group,
    }


value_report = build_value_report(df)
value_report["rank_ic"], value_report["group_report"]


这就是今天的目标产出：一个简化价值因子报告，包含：

- 因子分布
- 因子相关性
- Rank IC
- 分组表现

---

## 12. 实战中怎么使用价值报告？

一个价值因子报告至少要回答：

1. 因子有多少缺失？
2. 因子分布是否正常？
3. EP、BP、股息率是否高度重叠？
4. 因子方向是否符合预期？
5. 高价值组是否跑赢低价值组？
6. 结果是否被某个行业或少数极端样本驱动？

不要只看一张漂亮柱状图。  
价值因子最怕“便宜但有原因”。

---

## 13. 今天的知识图谱


In [ ]:
mindmap
  root((价值因子2))
    EP
      净利润除以市值
      PE倒数
      越高越便宜
      亏损需处理
    BP
      净资产除以市值
      PB倒数
      FamaFrench价值思路
      资产质量风险
    股息率
      分红除以市值
      现金回报
      成熟公司特征
      高股息可能低增长
    处理
      清洗负值
      去极值
      标准化
      综合打分
    报告
      分布
      相关性
      RankIC
      分组收益


---

## 14. 初学者最容易踩的 7 个坑

### 坑 1：不知道 EP 是 PE 的倒数

EP 越高通常越便宜，PE 越低通常越便宜。

### 坑 2：把负 EP 当正常高低排序

负 EP 来自亏损公司，需要单独处理。

### 坑 3：只看股息率

高股息可能来自稳定现金流，也可能来自股价大跌。

### 坑 4：忽略分红可持续性

一次性高分红不等于长期高股息。

### 坑 5：不看因子相关性

多个价值指标可能高度重复，合成时要知道自己在叠加什么。

### 坑 6：忽略行业

股息率、BP、EP 都可能有明显行业偏差。

### 坑 7：报告只写结论不写证据

报告至少要有分布、相关性、IC 和分组结果。

---

## 15. 今天的动手作业

### 作业 A：解释 EP/BP

用自己的话解释：

1. EP 和 PE 的关系。
2. BP 和 PB 的关系。
3. 为什么 EP/BP 更像天然正向因子。

### 作业 B：运行价值报告

运行本文代码，输出：

- 因子描述统计
- 因子相关矩阵
- Rank IC
- 5 分组报告

### 作业 C：观察股息率

查看股息率最高的 10 只股票，判断它们是否一定是好公司。

### 作业 D：修改合成权重

把综合分数改成：


value_score = 0.5 * value_ep + 0.3 * value_bp + 0.2 * value_dividend


观察分组收益是否变化。

---

## 16. 自测题

### 题 1

EP 的公式是什么？

答案：净利润 / 总市值。

### 题 2

BP 的公式是什么？

答案：净资产 / 总市值。

### 题 3

股息率为什么可以算价值因子？

答案：它衡量现金分红相对于市值的回报，越高代表价格相对现金分红越低。

### 题 4

EP 为负代表什么？

答案：通常代表公司亏损，不应直接当作普通估值排序。

### 题 5

价值因子报告至少应该包含哪些内容？

答案：分布、相关性、IC、分组收益，以及异常值和缺失情况说明。

---

## 17. 今日复盘模板


第 07 天复盘：价值因子 2

1. 我今天理解的 EP：

2. 我今天理解的 BP：

3. 我今天理解的股息率：

4. 我的价值报告核心结论：

5. 我发现的异常样本：

6. 我认为价值因子最大风险：

7. 明天学习质量因子前，我需要准备：


---

## 18. 明天预告：质量因子 1

价值因子问的是“便宜不便宜”。  
质量因子问的是：

> 这家公司是不是一门好生意？

明天会学习 ROE、ROA 和毛利率。

---

## 19. 一句话收尾

EP、BP、股息率把价值投资的直觉换成了更容易排序的语言。

> 花同样一元钱，买到更多盈利、更多净资产、更多现金分红，这就是价值因子的基本味道。

---

## 20. 仅供学习的提醒

本文所有示例使用模拟数据，仅用于解释价值因子报告方法，不构成任何投资建议。真实研究需要使用准确的财务披露时点、复权市值、现金分红数据、行业分类和样本外检验。

---

# 统一高质量增强模块

> 本增强模块用于把第 07 天课程统一提升到第 1-2 天那种“能直接学习、能直接运行、能直接复盘”的密度。前面的正文保留；下面是更完整的学习版。

## A. 今日任务重新聚焦

- 主题：价值因子2
- 必做：EP/BP
- 选做：股息率
- 目标产出：价值因子报告

今天真正要练成的不是“知道一个名词”，而是能把这个主题放进完整因子研究流水线：


原始数据
  ↓
因子构造
  ↓
预处理和对齐
  ↓
IC / ICIR / 分层回测
  ↓
形成可复用模块


你学习时可以一直问自己三句话：

1. 这个因子在经济含义上解释什么？
2. 这个因子在代码里如何被严格计算？
3. 这个因子是否真的经得起检验，而不是只在故事里成立？

## B. 一个更生动的直觉案例

PE/PB 是价格相对基本面的倍数；EP/BP 则反过来问：每一元价格买到了多少盈利和净资产。

这个例子背后的关键直觉是：

> 把估值倒过来看，往往更适合做正向因子。

因子研究不是把金融名词翻译成代码，而是把一个投资假设拆成可以被验证、被复现、被质疑的实验。

## C. 今日知识骨架


价值因子2
├── 输入数据
│   ├── 行情 / 财务 / 行业 / 市值等基础字段
│   └── 明确每个字段在当时是否可得
├── 因子定义
│   ├── 写清楚公式
│   ├── 写清楚方向
│   └── 写清楚缺失和异常值处理
├── 因子检验
│   ├── Rank IC
│   ├── ICIR
│   └── 分层回测
└── 目标产出
    └── 价值因子报告


## D. 完整 Python 实验

下面这段代码是一个自包含实验。你可以单独复制到 Notebook 里运行。它的目的不是模拟真实市场，而是把今天主题的计算口径、方向、检查方法串起来。


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(107)
n = 350
df = pd.DataFrame({
    "ticker": [f"S{i:03d}" for i in range(n)],
    "market_cap": rng.lognormal(10, 1, n),
    "net_income": rng.normal(900, 500, n),
    "book_equity": rng.normal(6000, 1800, n),
})
payout = np.clip(rng.normal(.25, .15, n), 0, .8)
df["dividend"] = np.maximum(df["net_income"], 0) * payout
df["ep"] = (df["net_income"] / df["market_cap"]).where(df["net_income"] > 0)
df["bp"] = (df["book_equity"] / df["market_cap"]).where(df["book_equity"] > 0)
df["dividend_yield"] = df["dividend"] / df["market_cap"]

def zscore(s):
    s = s.clip(s.quantile(.01), s.quantile(.99))
    return (s - s.mean()) / s.std()

for c in ["ep", "bp", "dividend_yield"]:
    df[f"value_{c}"] = zscore(df[c])
df["value_score"] = df[["value_ep", "value_bp", "value_dividend_yield"]].mean(axis=1)
print(df[["value_ep", "value_bp", "value_dividend_yield", "value_score"]].corr().round(3))


## E. 产出验收标准

完成今天课程后，你的 `价值因子报告` 至少应该满足：

1. 字段命名清晰，能看出日期、股票、因子值和标签含义。
2. 因子方向明确：值越大到底代表越好、越便宜、越强，还是越低风险。
3. 缺失值和异常值有处理口径，不把未知伪装成 0。
4. 至少有一段可重复运行的 Python 实验验证核心逻辑。
5. 能用 IC、ICIR 或分层回测中的至少一种方法做初步检查。
6. 能解释这个因子在真实研究里可能失效的原因。

如果这些检查没有过，不要急着进入下一天。因子研究里很多错误不是模型问题，而是最开始的口径、方向、对齐、缺失值处理出了问题。

## F. 常见坑深挖

### 坑 1：只记公式，不检查数据可得时点

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 2：因子方向写反，却直接进入 IC 和回测

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 3：把模拟数据里的漂亮结果当成真实市场规律

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 4：忽略缺失值、极端值和样本边界

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 5：只看单一指标，不做交叉验证

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 6：没有把目标产出封装成可复用函数

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。

## G. 强化练习

### 作业 A

用自己的话写出 `价值因子2` 的一句话定义，并标明它属于收益、风险、估值、质量、技术、流动性还是预处理模块。
### 作业 B

运行完整实验代码，记录输出结果，并解释每一列结果的金融含义。
### 作业 C

故意把因子方向取反，再重新计算结果，观察 IC 或分组表现如何变化。
### 作业 D

加入 5% 缺失值或 1% 极端值，测试你的处理逻辑是否仍然稳健。
### 作业 E

把今天的 `价值因子报告` 保存成一个可以被后续课程调用的函数或表格。

## H. 面试式自测

### 问：这个主题在因子研究流水线里处于哪一步？

答：它对应 `价值因子报告`，用于把原始数据转成后续 IC、ICIR、分层回测或多因子合成可以使用的中间产物。
### 问：最容易出现未来函数的地方在哪里？

答：通常出现在使用未来才披露的数据、未来价格、未来收益标签错位，或把全样本统计量用于历史截面。
### 问：为什么不能只看一个漂亮结果？

答：因为单次结果可能来自样本偶然、极端值、行业暴露、市值暴露或参数过拟合，需要多角度验证。
### 问：如何判断今天产出的模块可以进入下一步？

答：至少通过字段检查、方向检查、缺失异常检查、抽样手工验证和一个简单统计检验。

## I. 今日复盘模板


第 07 天复盘：价值因子2

1. 今天我能用一句话解释的核心概念：

2. 今天最重要的公式：

3. 代码里最容易写错的地方：

4. 我检查因子方向的方法：

5. 我检查缺失值和异常值的方法：

6. 如果把这个模块放进真实研究，我还缺什么数据：

7. 今天留下的一个问题：


## J. 和下一课的连接

下一课会继续沿着这条链路推进：前一天产出的字段或模块，会成为后一天检验、扩展或组合的输入。学习时不要把每天割裂开；真正的因子研究是一条流水线。

---

## K. 学习提醒

这一份课程仍然是教学材料，示例数据是模拟数据。真实研究需要处理真实数据源、可得时点、复权、停牌、交易成本、行业和市值暴露、样本外验证。
